---
title: "Check JAX data"
author: "Bo Yu, Si Liu, Wei Sun"
date: "`r Sys.Date()`"
output: 
  html_document:
    theme: journal
    highlight: tango
    toc: true
    toc_depth: 3
    toc_float:
      collapsed: true
      smooth_scroll: false
    number_sections: false
  df_print: paged
---



In [ ]:
knitr::opts_chunk$set(echo = TRUE)



## R libraries.



In [1]:
library(ggplot2)
library(ggrepel)
library(ggpubr)
library(stringr)
library(MASS)
library(RColorBrewer)

library(viridis)
library(ggpointdensity)
library(dplyr)
library(data.table)
library(readxl)

theme_set(theme_classic())

personal_path = "/fh/working/sun_w/sshen/MorPhiC"
path = "/fh/fast/sun_w/MorPhiC/data/MorPhiC_exchange_experiment/"
dat_path = file.path(path, "DRACC-processed/JAX_exchange_experiment_DRACC_processed_August2025/Tables")

metadata_path = file.path(path, "JAX/morphic_exchange_experiment")

read_header <- function(file_name, sheet){
  
  meta_header = read_excel(file_name, sheet = sheet, 
                         skip = 3, n_max = 1, col_names = FALSE)
  meta_header = as.character(meta_header)
  meta_header = gsub("_cell_line", "", meta_header, fixed = TRUE)
  meta_header = gsub("differentiated_product.", "", meta_header, fixed = TRUE)
  meta_header = gsub(".text", "", meta_header, fixed = TRUE)
  
  meta_header
}


Loading required package: viridisLite


Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last





## Read in meta data



In [2]:
meta_file = paste0(metadata_path, "/DPC_metadata_template_EGE_JAX-final.xlsx")

cl_header = read_header(meta_file, sheet = "Clonal cell line")
cl_header

cl = read_excel(meta_file, sheet = "Clonal cell line", skip = 5, col_names = FALSE)
names(cl) = cl_header
cl = as.data.frame(cl)
dim(cl)
cl[1:2,]

dc_header = read_header(meta_file, sheet = "Differentiated product")
dc_header

dc = read_excel(meta_file, sheet = "Differentiated product", 
                skip = 5, col_names = FALSE)
names(dc) = dc_header
dc = as.data.frame(dc)
dim(dc)
dc[1:2,]

table(dc$timepoint_value, useNA="ifany")

eas_header = read_header(meta_file, sheet = "Expression alteration")
eas_header

eas = read_excel(meta_file, sheet = "Expression alteration", 
                skip = 5, col_names = FALSE)
names(eas) = eas_header
eas = as.data.frame(eas)
dim(eas)
eas[1:2,]


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`


[1] "clonal.label"                        
 [2] "clonal.description"                  
 [3] "clonal.parental_name"                
 [4] "clonal.clone_id"                     
 [5] "clonal.type"                         
 [6] "clonal.zygosity"                     
 [7] "clonal.cell_line_generation_protocol"
 [8] "clonal.treatment_condition"          
 [9] "clonal.wt_control_status"            
[10] "expression_alteration.label"

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`


[1]  2 10

,clonal.label,clonal.description,clonal.parental_name,clonal.clone_id,clonal.type,clonal.zygosity,clonal.cell_line_generation_protocol,clonal.treatment_condition,clonal.wt_control_status,expression_alteration.label
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,MOK20022W-C01,KOLF2.2J GCM1 null by deletion of full coding region (KO),KOLF2.2J,C01,iPSC,Not applicable,Not applicable,Not applicable,KO,JAX_GCM1_Full_coding_length
2,MOK20022W-E04,KOLF2.2J GCM1 null by deletion of full coding region (KO),KOLF2.2J,E04,iPSC,Not applicable,Not applicable,Not applicable,KO,JAX_GCM1_Full_coding_length


...

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`


[1] "expression_alteration.label"                                      
 [2] "expression_alteration.parent_protocol_id"                         
 [3] "expression_alteration.method"                                     
 [4] "expression_alteration.genes.allele_specific"                      
 [5] "expression_alteration.genes.altered_gene_symbol"                  
 [6] "expression_alteration.genes.target_gene_hgnc_id"                  
 [7] "expression_alteration.genes.targeted_genomic_region"              
 [8] "expression_alteration.genes.expected_rna_alteration_phenotype"    
 [9] "expression_alteration.genes.expected_protein_alteration_phenotype"
[10] "expression_alteration.genes.editing_strategy"                     
[11] "expression_alteration.genes.altered_locus"                        
[12] "expression_alteration.genes.guide_sequence"

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`


[1]  1 12

,expression_alteration.label,expression_alteration.parent_protocol_id,expression_alteration.method,expression_alteration.genes.allele_specific,expression_alteration.genes.altered_gene_symbol,expression_alteration.genes.target_gene_hgnc_id,expression_alteration.genes.targeted_genomic_region,expression_alteration.genes.expected_rna_alteration_phenotype,expression_alteration.genes.expected_protein_alteration_phenotype,expression_alteration.genes.editing_strategy,expression_alteration.genes.altered_locus,expression_alteration.genes.guide_sequence
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,JAX_GCM1_Full_coding_length,dx.doi.org/10.17504/protocols.io.e6nvwb24wvmk/v1,CRISPR/Cas9 method,no,GCM1,HGNC:4197,Full coding region,RNA down-regulated,Protein off,full coding length,chr6:53127676-53145648,TGATAAGGTCAGGCCAGCCA;TAGTATTTCCACCCTCAGTA
NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA



## Check cell lines



In [3]:
any(duplicated(cl$clonal.label))
table(cl$clonal.type)
table(cl$clonal.parental_name)
table(cl$clonal.zygosity)
sort(table(cl$clonal.clone_id), decreasing = TRUE)

table(dc$clonal.label %in% cl$clonal.label)
table(cl$clonal.label %in% dc$clonal.label)
setdiff(dc$clonal.label, cl$clonal.label)
setdiff(cl$clonal.label, dc$clonal.label)

any(duplicated(dc$label))
table(table(dc$clonal.label))
table(dc$timepoint_value)
table(dc$timepoint_unit)
table(dc$final_timepoint)
table(dc$differentiated_product_protocol_id, 
      dc$model_system)

table(dc$timepoint_value, 
      dc$model_system)


[1] FALSE


iPSC 
   2 


KOLF2.2J 
       2 


Not applicable 
             2 


C01 E04 
  1   1 

...


 6 
62 


days 
  62 


yes 
 62 

                                                  
                                                   primitive syncytium
  dx.doi.org/10.17504/protocols.io.q26g7mwekgwz/v1                  62

   
    primitive syncytium
  6                  62


## Check Expression alteration strategy



In [4]:
any(duplicated(eas$expression_alteration.label))
setdiff(eas$expression_alteration.label, cl$expression_alteration.label)
setdiff(cl$expression_alteration.label, eas$expression_alteration.label)

table(eas$expression_alteration.parent_protocol_id)
table(eas$expression_alteration.method)
table(eas$expression_alteration.genes.allele_specific)
table(eas$expression_alteration.parent_protocol_id, 
      eas$expression_alteration.genes.altered_gene_symbol, useNA="ifany")
table(eas$expression_alteration.parent_protocol_id, 
      eas$expression_alteration.genes.targeted_genomic_region, useNA="ifany")
table(eas$expression_alteration.genes.targeted_genomic_region, 
      eas$expression_alteration.genes.editing_strategy, useNA="ifany")
table(eas$expression_alteration.genes.editing_strategy, 
      eas$expression_alteration.genes.expected_rna_alteration_phenotype, useNA="ifany")
table(eas$expression_alteration.genes.editing_strategy, 
      eas$expression_alteration.genes.expected_protein_alteration_phenotype, useNA="ifany")


[1] FALSE

character(0)

character(0)


dx.doi.org/10.17504/protocols.io.e6nvwb24wvmk/v1 
                                               1 


CRISPR/Cas9 method 
                 1 

...

                                                  
                                                   GCM1
  dx.doi.org/10.17504/protocols.io.e6nvwb24wvmk/v1    1

                                                  
                                                   Full coding region
  dx.doi.org/10.17504/protocols.io.e6nvwb24wvmk/v1                  1

                    
                     full coding length
  Full coding region                  1

                    
                     RNA down-regulated
  full coding length                  1

                    
                     Protein off
  full coding length           1


## Extract KO strategy information



In [5]:
unique(dc$description)

dc <- dc %>%
  mutate(
    ko_strategy = case_when(
      str_detect(description, "knockout cell line, .* KO from JAX") ~ "KO_JAX",
      str_detect(description, "knockout cell line, .* KO from MSK") ~ "KO_MSK",
      str_detect(description, "^KOLF2\\.2J differentiated\\b") & str_detect(label, "^(JAX_|MSK_)") ~ "CTRL_KO_WT",
      str_detect(description, "^KOLF2\\.2J differentiated\\b") & str_detect(label, "^NW_") ~ "CTRL_AID_WT",
      str_detect(description, "^KOLF2\\.2J differentiated\\b") & str_detect(label, "^UCSF_") ~ "CTRL_CRISPRi_WT",
      str_detect(description, "auxin induced degron null") ~ "AID_NW",
      str_detect(description, "arboring OsTIR1 as a control for AID system") ~ "CTRL_AID_OsTIR1",
      str_detect(description, "derived DMSO control for degron null") ~ "CTRL_AID_DMSO",
      str_detect(description, "CRISPRi generated .* null") ~ "CRISPRi_UCSF",
      str_detect(description, "CRISPRi transgene in the AAVS1") ~ "CTRL_CRISPRi_AAVS1",
      str_detect(description, "derived DMSO control for CRISPR") ~ "CTRL_CRISPRi_DMSO",
      TRUE ~ NA_character_
    ),
    ko_gene = coalesce(
      str_match(description, "(?<=derived knockout cell line, )([A-Za-z0-9-]+)(?=\\s+KO\\b)")[,2],
      str_match(description, "(?<=degron null of )([A-Za-z0-9-]+)")[,2],
      str_match(description, "(?<=CRISPRi generated )([A-Za-z0-9-]+)(?=\\s+null\\b)")[,2]
    ),
    ko_gene = ifelse(is.na(ko_gene), "WT", ko_gene)
  )

table(dc$ko_strategy, dc$ko_gene, useNA = 'ifany')

table(substr(dc$description, 1, 53), dc$ko_strategy, useNA = 'ifany')


[1] "KOLF2.2J derived knockout cell line, GCM1 KO from JAX, differentiated to the trophoblast lineage and favoring primitive syncytium formation"             
[2] "KOLF2.2J derived knockout cell line, GCM1 KO from MSK, differentiated to the trophoblast lineage and favoring primitive syncytium formation"             
[3] "KOLF2.2J differentiated to the trophoblast lineage and favoring primitive syncytium formation"                                                           
[4] "KOLF2.2J derived auxin induced degron null of GCM1 differentiated to the trophoblast lineage and favoring primitive syncytium formation"                 
[5] "KOLF2.2J arboring OsTIR1 as a control for AID system differentiated to the trophoblast lineage and favoring primitive syncytium formation"               
[6] "KOLF2.2J derived DMSO control for degron null GCM1 differentiated to the trophoblast lineage and favoring primitive syncytium formation"                 
[7] "KOLF2.2J derived CRISPRi generated GCM1 null differentiated to the trophoblast lineage and favoring primitive syncytium formation"                       
[8] "KOLF2.2J arboring CRISPRi transgene in the AAVS1 locus as a control differentiated to the trophoblast lineage and favoring primitive syncytium formation"
[9] "KOLF2.2J derived DMSO control for CRISPR generated GCM1 null differentiated to the trophoblast lineage and favoring primitive syncytium formation"

                    
                     GCM1 WT
  AID_NW                6  0
  CRISPRi_UCSF          6  0
  CTRL_AID_DMSO         0  6
  CTRL_AID_OsTIR1       0  6
  CTRL_AID_WT           0  5
  CTRL_CRISPRi_AAVS1    0  6
  CTRL_CRISPRi_DMSO     0  6
  CTRL_CRISPRi_WT       0  6
  CTRL_KO_WT            0  3
  KO_JAX                6  0
  KO_MSK                6  0

                                                       
                                                        AID_NW CRISPRi_UCSF
  KOLF2.2J arboring CRISPRi transgene in the AAVS1 locu      0            0
  KOLF2.2J arboring OsTIR1 as a control for AID system       0            0
  KOLF2.2J derived auxin induced degron null of GCM1 di      6            0
  KOLF2.2J derived CRISPRi generated GCM1 null differen      0            6
  KOLF2.2J derived DMSO control for CRISPR generated GC      0            0
  KOLF2.2J derived DMSO control for degron null GCM1 di      0            0
  KOLF2.2J derived knockout cell line, GCM1 KO from JAX      0            0
  KOLF2.2J derived knockout cell line, GCM1 KO from MSK      0            0
  KOLF2.2J differentiated to the trophoblast lineage an      0            0
                                                       
                                                        CTRL_AID_DMSO
  KOLF2.2J arboring CRISPRi transgene in the AAVS1 locu   



## Read in library preparation information



In [6]:
lib_header = read_header(meta_file, sheet = "Library preparation")
lib_header

lib = read_excel(meta_file, sheet = "Library preparation", 
                skip = 5, col_names = FALSE)
dim(lib)
names(lib) = lib_header
lib = as.data.frame(lib)
dim(lib)
lib[1:2,]

any(duplicated(lib$label))
setdiff(dc$label, lib$label)
setdiff(lib$label, dc$label)


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`


[1] "library_preparation.label"                          
 [2] "library_preparation.description"                    
 [3] "library_preparation.library_preparation_protocol_id"
 [4] "library_preparation.average_fragment_size"          
 [5] "library_preparation.input_amount_value"             
 [6] "library_preparation.input_amount_unit"              
 [7] "library_preparation.concentration_value"            
 [8] "library_preparation.concentration_unit"             
 [9] "library_preparation.total_yield_value"              
[10] "library_preparation.total_yield_unit"               
[11] "library_preparation.cdna_pcr_cycles"                
[12] "library_preparation.pcr_cycles_for_indexing"        
[13] "label"

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...12`
• `` -> `...13`


[1] 62 13

[1] 62 13

,library_preparation.label,library_preparation.description,library_preparation.library_preparation_protocol_id,library_preparation.average_fragment_size,library_preparation.input_amount_value,library_preparation.input_amount_unit,library_preparation.concentration_value,library_preparation.concentration_unit,library_preparation.total_yield_value,library_preparation.total_yield_unit,library_preparation.cdna_pcr_cycles,library_preparation.pcr_cycles_for_indexing,label
,<chr>,<lgl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>
1,S26_GT25-02971,NA,dx.doi.org/10.17504/protocols.io.3byl4z39ovo5/v1,480,100,ng,8.98,ng/uL,179.6,ng,0,12,JAX_C01_Rep1
2,S21_GT25-02972,NA,dx.doi.org/10.17504/protocols.io.3byl4z39ovo5/v1,469,100,ng,7.28,ng/uL,145.6,ng,0,12,JAX_C01_Rep2


[1] FALSE

character(0)

character(0)


## Read in sequence file information



In [7]:
seq_header = read_header(meta_file, sheet = "Sequence file")
seq_header

seq = read_excel(meta_file, sheet = "Sequence file", 
                skip = 5, col_names = FALSE)
names(seq) = seq_header
seq = as.data.frame(seq)
seq$file_id = gsub("_R(1|2)_001.fastq.gz", "", 
                   seq$sequence_file.label)
dim(seq)
seq[1:2,]

seq = unique(seq[,c("file_id", "library_preparation.label", 
                    "sequence_file.run_id")])
dim(seq)
seq[1:2,]
length(unique(seq$library_preparation.label))

table(seq$sequence_file.run_id)


New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`


[1] "sequence_file.label"       "sequence_file.extension"  
[3] "sequence_file.lane_index"  "sequence_file.read_index" 
[5] "sequence_file.read_length" "library_preparation.label"
[7] "sequence_file.run_id"

New names:
• `` -> `...1`
• `` -> `...2`
• `` -> `...3`
• `` -> `...4`
• `` -> `...5`
• `` -> `...6`
• `` -> `...7`


[1] 124   8

,sequence_file.label,sequence_file.extension,sequence_file.lane_index,sequence_file.read_index,sequence_file.read_length,library_preparation.label,sequence_file.run_id,file_id
,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>
1,JAX_C01_Rep1_GT25-02971_TGTACCGTNNNNNNNNN-CCTAGAGA_S26_L002_R1_001.fastq.gz,fastq.gz,L002,read1,150,S26_GT25-02971,20250616_GT25-RobsonP-99,JAX_C01_Rep1_GT25-02971_TGTACCGTNNNNNNNNN-CCTAGAGA_S26_L002
2,JAX_C01_Rep2_GT25-02972_TGACTGACNNNNNNNNN-CAGAACTG_S21_L002_R1_001.fastq.gz,fastq.gz,L002,read1,150,S21_GT25-02972,20250616_GT25-RobsonP-99,JAX_C01_Rep2_GT25-02972_TGACTGACNNNNNNNNN-CAGAACTG_S21_L002


[1] 62  3

,file_id,library_preparation.label,sequence_file.run_id
,<chr>,<chr>,<chr>
1,JAX_C01_Rep1_GT25-02971_TGTACCGTNNNNNNNNN-CCTAGAGA_S26_L002,S26_GT25-02971,20250616_GT25-RobsonP-99
2,JAX_C01_Rep2_GT25-02972_TGACTGACNNNNNNNNN-CAGAACTG_S21_L002,S21_GT25-02972,20250616_GT25-RobsonP-99


[1] 62


20250616_GT25-RobsonP-99 
                      62 


## Merge tabs to make a flat metedata table



In [8]:
intersect(names(dc), names(lib))
table(dc$label%in%lib$label, 
      useNA="ifany")
table(lib$label%in%dc$label, 
      useNA="ifany")

meta = merge(dc, lib)
dim(meta)
meta[1:2,]

intersect(names(meta), names(seq))

any(duplicated(meta$library_preparation.label))
any(duplicated(seq$library_preparation.label))

table(meta$library_preparation.label%in%seq$library_preparation.label, 
      useNA="ifany")
table(seq$library_preparation.label%in%meta$library_preparation.label, 
      useNA="ifany")

meta = merge(meta, seq)
dim(meta)
meta[1:2,]

update_name <- function(df1, old_name, new_name){
  ww1 = which(names(df1) == old_name)
  stopifnot(length(ww1) == 1)
  names(df1)[ww1] = new_name
  df1
}

meta = update_name(meta, "sequence_file.run_id", "run_id")


[1] "label"


TRUE 
  62 


TRUE 
  62 

[1] 62 24

,label,description,clonal.label,differentiated_product_protocol_id,model_system,timepoint_value,timepoint_unit,final_timepoint,treatment_condition,wt_control_status,⋯,library_preparation.library_preparation_protocol_id,library_preparation.average_fragment_size,library_preparation.input_amount_value,library_preparation.input_amount_unit,library_preparation.concentration_value,library_preparation.concentration_unit,library_preparation.total_yield_value,library_preparation.total_yield_unit,library_preparation.cdna_pcr_cycles,library_preparation.pcr_cycles_for_indexing
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
1,JAX_C01_Rep1,"KOLF2.2J derived knockout cell line, GCM1 KO from JAX, differentiated to the trophoblast lineage and favoring primitive syncytium formation",MOK20022W-C01,dx.doi.org/10.17504/protocols.io.q26g7mwekgwz/v1,primitive syncytium,6,days,yes,Not applicable,KO,⋯,dx.doi.org/10.17504/protocols.io.3byl4z39ovo5/v1,480,100,ng,8.98,ng/uL,179.6,ng,0,12
2,JAX_C01_Rep2,"KOLF2.2J derived knockout cell line, GCM1 KO from JAX, differentiated to the trophoblast lineage and favoring primitive syncytium formation",MOK20022W-C01,dx.doi.org/10.17504/protocols.io.q26g7mwekgwz/v1,primitive syncytium,6,days,yes,Not applicable,KO,⋯,dx.doi.org/10.17504/protocols.io.3byl4z39ovo5/v1,469,100,ng,7.28,ng/uL,145.6,ng,0,12


...

[1] FALSE


TRUE 
  62 


TRUE 
  62 

[1] 62 26

,library_preparation.label,label,description,clonal.label,differentiated_product_protocol_id,model_system,timepoint_value,timepoint_unit,final_timepoint,treatment_condition,⋯,library_preparation.input_amount_value,library_preparation.input_amount_unit,library_preparation.concentration_value,library_preparation.concentration_unit,library_preparation.total_yield_value,library_preparation.total_yield_unit,library_preparation.cdna_pcr_cycles,library_preparation.pcr_cycles_for_indexing,file_id,sequence_file.run_id
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,⋯,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,S21_GT25-02972,JAX_C01_Rep2,"KOLF2.2J derived knockout cell line, GCM1 KO from JAX, differentiated to the trophoblast lineage and favoring primitive syncytium formation",MOK20022W-C01,dx.doi.org/10.17504/protocols.io.q26g7mwekgwz/v1,primitive syncytium,6,days,yes,Not applicable,⋯,100,ng,7.28,ng/uL,145.6,ng,0,12,JAX_C01_Rep2_GT25-02972_TGACTGACNNNNNNNNN-CAGAACTG_S21_L002,20250616_GT25-RobsonP-99
2,S22_GT25-02976,JAX_E04_Rep3,"KOLF2.2J derived knockout cell line, GCM1 KO from JAX, differentiated to the trophoblast lineage and favoring primitive syncytium formation",MOK20022W-E04,dx.doi.org/10.17504/protocols.io.q26g7mwekgwz/v1,primitive syncytium,6,days,yes,Not applicable,⋯,100,ng,7.44,ng/uL,148.8,ng,0,12,JAX_E04_Rep3_GT25-02976_TTCCTGTGNNNNNNNNN-CTCCTGAA_S22_L002,20250616_GT25-RobsonP-99


In [9]:
# intersect(names(meta), names(cl))

# table(meta$clonal.label %in% cl$clonal.label)
# table(cl$clonal.label %in% meta$clonal.label)
# setdiff(meta$clonal.label, cl$clonal.label)
# setdiff(cl$clonal.label, meta$clonal.label)

# meta = merge(meta, cl)
# dim(meta)
# meta[1:2,]

table(meta$ko_strategy, useNA = "ifany")
meta$ko_strategy[which(is.na(meta$ko_strategy))] = "WT"

table(meta$ko_gene, useNA = "ifany")
meta$ko_gene[which(is.na(meta$ko_gene))] = "WT"

meta$model_organ = meta$model_system
meta$model_system = NA
meta$model_system = str_extract(meta$label, "^[^-]+")
table(meta$model_system, meta$model_organ)

table(meta$model_system, meta$ko_gene, useNA = "ifany")

fwrite(meta, file = file.path(personal_path, "JAX_meta_data.tsv"), sep="\t")



            AID_NW       CRISPRi_UCSF      CTRL_AID_DMSO    CTRL_AID_OsTIR1 
                 6                  6                  6                  6 
       CTRL_AID_WT CTRL_CRISPRi_AAVS1  CTRL_CRISPRi_DMSO    CTRL_CRISPRi_WT 
                 5                  6                  6                  6 
        CTRL_KO_WT             KO_JAX             KO_MSK 
                 3                  6                  6 


GCM1   WT 
  24   38 

                       
                        primitive syncytium
  JAX_C01_Rep1                            1
  JAX_C01_Rep2                            1
  JAX_C01_Rep3                            1
  JAX_E04_Rep1                            1
  JAX_E04_Rep2                            1
  JAX_E04_Rep3                            1
  JAX_MSK_WT1                             1
  JAX_MSK_WT2                             1
  JAX_MSK_WT3                             1
  MSK_Clone1_Rep1                         1
  MSK_Clone1_Rep2                         1
  MSK_Clone1_Rep3                         1
  MSK_Clone2_Rep1                         1
  MSK_Clone2_Rep2                         1
  MSK_Clone2_Rep3                         1
  NW_C20_DMSO_Rep1                        1
  NW_C20_DMSO_Rep2                        1
  NW_C20_DMSO_Rep3                        1
  NW_C20_IAA_Rep1                         1
  NW_C20_IAA_Rep2                         1
  NW_C20_IAA_Rep3                         1
  NW_C43

                       
                        GCM1 WT
  JAX_C01_Rep1             1  0
  JAX_C01_Rep2             1  0
  JAX_C01_Rep3             1  0
  JAX_E04_Rep1             1  0
  JAX_E04_Rep2             1  0
  JAX_E04_Rep3             1  0
  JAX_MSK_WT1              0  1
  JAX_MSK_WT2              0  1
  JAX_MSK_WT3              0  1
  MSK_Clone1_Rep1          1  0
  MSK_Clone1_Rep2          1  0
  MSK_Clone1_Rep3          1  0
  MSK_Clone2_Rep1          1  0
  MSK_Clone2_Rep2          1  0
  MSK_Clone2_Rep3          1  0
  NW_C20_DMSO_Rep1         0  1
  NW_C20_DMSO_Rep2         0  1
  NW_C20_DMSO_Rep3         0  1
  NW_C20_IAA_Rep1          1  0
  NW_C20_IAA_Rep2          1  0
  NW_C20_IAA_Rep3          1  0
  NW_C43_DMSO_Rep1         0  1
  NW_C43_DMSO_Rep2         0  1
  NW_C43_DMSO_Rep3         0  1
  NW_C43_IAA_Rep1          1  0
  NW_C43_IAA_Rep2          1  0
  NW_C43_IAA_Rep3          1  0
  NW_Parent_DMSO_Rep1      0  1
  NW_Parent_DMSO_Rep2      0  1
  NW_Parent_DMSO


## Check count data

### genesCounts.csv



In [10]:
cts = fread(file.path(dat_path, "genesCounts.csv"), data.table = FALSE)
dim(cts)
cts[1:2, c(1:2, (ncol(cts)-1):ncol(cts))]
any(is.na(cts))

setdiff(names(cts)[-1], meta$file_id)
setdiff(meta$file_id, names(cts)[-1])


[1] 36601    63

,Name,NW_Parent_IAA_Rep3_GT25-02994_GTTAAGGCNNNNNNNNN-ACCTTCGA_S44_L002,NW_C20_IAA_Rep3_GT25-02988_CCGGAATTNNNNNNNNN-ACCGAATG_S28_L002,NW_WT_IAA_Rep2_GT25-02996_TAACGAGGNNNNNNNNN-TAGTCTCG_S78_L003
,<chr>,<int>,<int>,<int>
1,ENSG00000268674,0,0,0
2,ENSG00000271254,932,1101,866


[1] FALSE

character(0)

character(0)

In [11]:
gc()
sessionInfo()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1342670,71.8,2512294,134.2,2512294,134.2
Vcells,4001966,30.6,8388608,64.0,7026989,53.7


R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/sshen2/.conda/envs/r/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] readxl_1.5.0         data.table_1.18.4    dplyr_1.2.1         
 [4] ggpointdensity_0.2.1 viridis_0.6.5        viridisLite_0.4.3   
 [7] RColorBrewer_1.1-3   MASS_7.3-65          stringr_1.6.0       
[10] ggpubr_1.0.0         ggrepel_0.9.8        ggplot2_4.0.3       

loaded via a namespac